In [1]:
import numpy as np
import os
import json

In [2]:
path = "D:/GAR/data/processed/myogym"
latest_directory = sorted(os.listdir(path))[-1]
print(f"Loading data from: {latest_directory}")
files = os.listdir(os.path.join(path, latest_directory))
print(f"Files in the directory: {files}")

Loading data from: 20260416_142923
Files in the directory: ['metadata.json', 'myogym_test.npz', 'myogym_train.npz', 'myogym_val.npz']


In [3]:
# load test data
test_data = np.load(os.path.join(path, latest_directory, "myogym_test.npz"), allow_pickle=True)
# load metadata json
with open(os.path.join(path, latest_directory, "metadata.json"), "r") as f:
    metadata = json.load(f)

metadata

{'date': '2026-04-16T14:29:23.426424',
 'dataset': 'myogym',
 'normalization_strategy': None,
 'window_size': 100,
 'window_step': 50,
 'train_ratio': 0.8,
 'val_ratio': 0.2,
 'seed': 42,
 'transform_units': True,
 'units': {'acceleration': 'm/s^2', 'rotation': 'rad/s'},
 'activity_mapping': {'0': 'No activity identified',
  '1': 'Seated Cable Rows',
  '2': 'One-Arm Dumbbell Row',
  '3': 'Wide-Grip Pulldown Behind The Neck',
  '4': 'Bent Over Barbell Row',
  '5': 'Reverse Grip Bent-Over Row',
  '6': 'Wide-Grip Front Pulldown',
  '7': 'Bench Press',
  '8': 'Incline Dumbbell Flyes',
  '9': 'Incline Dumbbell Press',
  '10': 'Dumbbell Flyes',
  '11': 'Pushups',
  '12': 'Leverage Chest Press',
  '13': 'Close-Grip Barbell Bench Press',
  '14': 'Bar Skullcrusher',
  '15': 'Triceps Pushdown',
  '16': 'Bench Dip / Dip',
  '17': 'Overhead Triceps Extension',
  '18': 'Tricep Dumbbell Kickback',
  '19': 'Spider Curl',
  '20': 'Dumbbell Alternate Bicep Curl',
  '21': 'Incline Hammer Curl',
  '22': 

In [4]:
x_test = test_data["x"]
y_test = test_data["y"]
meta_test = test_data["meta"]
x_test.shape, y_test.shape, meta_test.shape

((7216, 100, 6), (7216,), (7216,))

In [5]:
from keras import layers, models

In [6]:
def make_model(T=100, C=6):
    model = models.Sequential([
        layers.Input(shape=(T, C)),

        layers.Conv1D(filters=32, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=64, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=64, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.Conv1D(filters=32, kernel_size=10, padding="same"),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPool1D(2),
        layers.Dropout(rate=0.25),

        layers.GlobalAveragePooling1D(),
        layers.Dense(1, activation="sigmoid")
    ])
    return model

In [7]:
model = make_model()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 100, 32)        │         1,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 50, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 25, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 25, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 12, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 12, 32)         │        20,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 6, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 6, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 84,833 (331.38 KB)

 Trainable params: 84,449 (329.88 KB)

 Non-trainable params: 384 (1.50 KB)

In [8]:
from sklearn.metrics import classification_report

In [9]:
p_test = model.predict(x_test)
y_test_pred = (p_test >= 0.5).astype(int)
print(classification_report(y_test, y_test_pred, digits=3))


226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

           0      0.000     0.000     0.000      5892
           1      0.183     1.000     0.310      1324

    accuracy                          0.183      7216
   macro avg      0.092     0.500     0.155      7216
weighted avg      0.034     0.183     0.057      7216



d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\GAR\envs\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
